# Puncak perjalanan Nataru: arus berangkat atau arus balik?

Setiap akhir tahun beritanya sama: "puncak arus mudik diperkirakan hari ini". Saya penasaran, kalau data perjalanannya dibuka langsung, seperti apa bentuk gelombang Nataru itu sebenarnya: satu lonjakan besar menjelang 25 Desember, atau sesuatu yang lain? Dan apakah orang benar-benar menyiapkannya jauh-jauh hari, atau baru mencari tiket mepet keberangkatan?

In [1]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tools import gaya

gaya.terapkan()

# A) perjalanan nyata: Posko Nataru 2024/25 Kemenhub
k = pd.read_csv("data/pergerakan-nataru-2024-25.csv", parse_dates=["tanggal"])
k["h"] = (k["tanggal"] - pd.Timestamp("2024-12-25")).dt.days
harian = k.groupby("h")["penumpang"].sum()

# B) mencari tiket: indeks Google Trends harian, satu permintaan per musim
FRASA = ["tiket kereta api", "tiket pesawat", "tiket bus"]
MUSIM = list(range(2019, 2026))


def muat_trends(musim):
    d = pd.read_csv(f"data/mentah/trends/trends-harian-{musim}.csv", parse_dates=["date"])
    natal = pd.Timestamp(f"{musim}-12-25")
    d["rel"] = (d["date"] - natal).dt.days
    return d.set_index("rel")


print("kemenhub:", len(k), "baris | h", k["h"].min(), "s.d.", k["h"].max())
print("moda:", sorted(k["moda"].unique()))
print("trends:", len(MUSIM), "musim x", len(muat_trends(2024)), "hari x", len(FRASA), "frasa")

kemenhub: 105 baris | h -10 s.d. 10
moda: ['Angkutan Jalan', 'Angkutan Laut', 'Angkutan Perkeretaapian', 'Angkutan SDP', 'Angkutan Udara']
trends: 7 musim x 212 hari x 3 frasa


## Data & cara ukur

Dua sumber terbuka yang perannya sengaja dibedakan. Untuk **perjalanan nyata**:
data harian Posko Nataru 2024/25 Kementerian Perhubungan (katalog HUBNET), 5 moda x 21 hari (H-10 s.d. H+10, 15 Desember 2024 sampai 4 Januari 2025), hasil pantauan posko terpadu yang direkonsiliasi. Satu musim memang; niat memesan tiket bisa dilihat lebih jauh di sumber kedua.

Untuk **niat**: 
indeks Google Trends membaca tujuh musim. Dicatat jujur sejak awal: indeks Trends adalah proporsi pencarian (0-100), bukan orang; tiap musim dinormalkan server, sehingga antar musim hanya boleh dibaca sebagai lipatan terhadap dasar musimnya sendiri (rata-rata September-pertengahan November). Frasa yang dipakai: "tiket kereta api", "tiket pesawat", "tiket bus". Pencarian penanda niat, bukan perjalanan; dua sumber ini tidak diperas jadi satu angka. Panen 24 September 2026, rincian dan catatan panen di `data/README.md`.

In [2]:
# cakupan dan pemeriksaan dasar
print("--- perjalanan nyata: total per moda dan puncak hariannya")
for moda, d in k.groupby("moda"):
    i = d["penumpang"].idxmax()
    print(f"  {moda:27s} total {int(d['penumpang'].sum()):>12,} | puncak H{d.loc[i,'h']:+d} = {int(d.loc[i,'penumpang']):,}")
print(f"  TOTAL seluruh moda        {int(k['penumpang'].sum()):>12,}")
print("baris berganda (moda,tanggal):", k.duplicated(["moda", "tanggal"]).sum())
print()
print("--- trends: jumlah hari per musim", {m: len(muat_trends(m)) for m in MUSIM})

--- perjalanan nyata: total per moda dan puncak hariannya
  Angkutan Jalan              total    3,934,367 | puncak H+4 = 230,527
  Angkutan Laut               total    1,750,511 | puncak H-3 = 102,286
  Angkutan Perkeretaapian     total    4,357,747 | puncak H+4 = 245,982
  Angkutan SDP                total    2,917,681 | puncak H-3 = 178,599
  Angkutan Udara              total    5,359,093 | puncak H-3 = 301,488
  TOTAL seluruh moda          18,319,399
baris berganda (moda,tanggal): 0



--- trends: jumlah hari per musim {2019: 213, 2020: 212, 2021: 212, 2022: 212, 2023: 213, 2024: 212, 2025: 212}


## Pembedahan

Gambar pertama membuka bentuk gelombangnya: bukan satu lonjakan, melainkan teras tinggi bertulang dua puncak. Garis tebal seluruh moda digabung; garis tipis memperlihatkan tiap moda bergerak serempak.

In [3]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Gelombang Nataru: bukan satu lonjakan, tapi dua puncak",
        "Puncak pergi H-3 dan puncak balik H+4 masing-masing 1,01 juta penumpang; "
        "hari tersepi jendelanya 0,62 juta. Di tengahnya ada cekungan tahun baru.",
        "Posko Nataru 2024/25 Kemenhub, 5 moda angkutan umum, harian H-10 s.d. H+10",
        int(len(k)),
    )
    for moda, d in k.groupby("moda"):
        s = d.set_index("h")["penumpang"].sort_index() / 1000
        ax.plot(s.index, s.values, color=gaya.INK2, lw=0.9, alpha=0.5, zorder=2)
    tot = harian.sort_index() / 1000
    ax.plot(tot.index, tot.values, color=gaya.AKSEN, lw=2.4, zorder=3)
    ax.fill_between(tot.index, 0, tot.values, color=gaya.AKSEN, alpha=0.08, zorder=1)
    h_pergi = int(harian.loc[:0].idxmax())
    h_balik = int(harian.loc[1:].idxmax())
    ax.annotate(f"pergi: puncak H{h_pergi:+d}\n{harian[h_pergi]/1e6:,.2f} jt".replace(",", "."),
                xy=(h_pergi, harian[h_pergi]/1000), xytext=(h_pergi - 6.8, harian[h_pergi]/1000*0.98),
                fontsize=8.8 * fs, color=gaya.INK,
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9))
    ax.annotate(f"balik: puncak H+{h_balik}\n{harian[h_balik]/1e6:,.2f} jt".replace(",", "."),
                xy=(h_balik, harian[h_balik]/1000), xytext=(h_balik + 0.6, harian[h_balik]/1000*1.00),
                fontsize=8.8 * fs, color=gaya.INK,
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9))
    ax.axvline(0, color=gaya.INK2, ls=":", lw=0.9)
    ax.text(0.3, tot.min() * 0.94, "Natal", fontsize=8.3 * fs, color=gaya.INK2)
    ax.axvline(7, color=gaya.INK2, ls=":", lw=0.9)
    ax.text(6.9, tot.min() * 0.90, "1 Jan", fontsize=8.3 * fs, color=gaya.INK2, ha="right")
    ax.set_ylabel("penumpang per hari (ribu orang)", fontsize=9.5 * fs)
    ax.set_xlabel("hari terhadap 25 Des (H±); tebal: semua moda, tipis: per moda", fontsize=9.5 * fs)
    ax.grid(True, axis="y"); ax.grid(False, axis="x")
    gaya.simpan(fig, "01-gelombang-aktual", mode)


for mode in gaya.MODE:
    grafik_1(mode)

Lalu sejak kapan orang berniat pergi? Gambar kedua membaca pencarian tiga frasa tiket di tujuh musim, dinyatakan sebagai lipatan terhadap dasar musimnya sendiri. Puncak pencariannya ternyata sangat mepet dengan perjalanan nyata di gambar pertama, bukan berminggu-minggu sebelumnya.

In [4]:
JENDELA_DASAR = (-110, -40)   # Sep s.d. mid Nov: masa tenang tiap permintaan musiman


def lipatan_musim(term):
    per = {}
    for m in MUSIM:
        d = muat_trends(m)[term]
        dasar = d.loc[JENDELA_DASAR[0]:JENDELA_DASAR[1]].mean()
        per[m] = d / dasar
    return pd.DataFrame(per)


LIPAT = {t: lipatan_musim(t) for t in FRASA}

def grafik_2(mode):
    m_ = gaya.MODE[mode]
    fs = m_["fs"]
    fig_ukuran = (m_["figsize"][0] * 2.7, m_["figsize"][1])
    fig, axs = plt.subplots(1, 3, figsize=fig_ukuran, sharey=True)
    fig.suptitle("Niat mencari tiket: naik 1,6-2,9 kali dasar, puncaknya mepet berangkat",
                 x=0.02, ha="left", fontsize=13 * fs, fontweight="bold", color=gaya.INK)
    fig.text(0.02, 0.90,
             "Tiap panel satu frasa; garis tipis tiap musim, tebal median 7 musim. "
             "Garis putus-putus vertikal: hari puncak pencarian (median).",
             fontsize=10 * fs, color=gaya.INK2)
    fig.text(0.02, -0.05,
             "Sumber: Google Trends (indeks relatif 0-100, dasar Sep-Nov sendiri), 7 musim 2019/20-2025/26 "
             f"| n = {gaya.idn(len(MUSIM) * 212 * len(FRASA), 0)}",
             fontsize=8 * fs, color=gaya.INK2)
    hk = [(-55, "1 Nov"), (-34, "22 Nov"), (-13, "12 Des"), (0, "25 Des"), (12, "6 Jan")]
    for ax, term, warna in zip(axs, FRASA, [gaya.AKSEN, gaya.INK, gaya.INK2]):
        df5 = LIPAT[term].loc[-55:12].rolling(3, min_periods=2, center=True).mean()
        for m in MUSIM:
            ax.plot(df5.index, df5[m], color=gaya.INK2, lw=0.8, alpha=0.4, zorder=2)
        med = df5.median(axis=1)
        ax.plot(med.index, med.values, color=warna, lw=2.0, zorder=3)
        wins = df5.loc[-7:-1]
        pk = [wins[m].idxmax() for m in MUSIM if pd.notna(wins[m].max())]
        ax.axvline(np.median(pk), color=warna, ls=":", lw=0.9)
        ax.axhline(1, color=gaya.GRID, lw=0.6)
        ax.text(0.03, 0.95, f"{term}\npuncak median H{np.median(pk):+.0f}",
                transform=ax.transAxes, fontsize=8.6 * fs, color=warna, va="top")
        ax.set_xticks([x for x, _ in hk])
        ax.set_xticklabels([lab for _, lab in hk], fontsize=8 * fs)
        ax.grid(True, axis="y"); ax.grid(False, axis="x")
        for sisi in ("top", "right"):
            ax.spines[sisi].set_visible(False)
    axs[0].set_ylabel("lipatan vs dasar musim", fontsize=9.5 * fs)
    fig.subplots_adjust(top=0.80, wspace=0.08, left=0.06, right=0.99)
    gaya.simpan(fig, "02-niat-tiket", mode)


for mode in gaya.MODE:
    grafik_2(mode)

## Temuan

> Gelombang Nataru bukan satu lonjakan: ia teras tiga pekan bertulang dua puncak, arus pergi di H-3 lalu arus balik di H+4 (masing-masing 1,01 juta penumpang/hari, Posko Nataru 2024/25), dengan cekungan pergantian tahun di tengahnya. Dan orang ternyata tidak menyiapkan Nataru berpekan-pekan: pencarian tiket konsisten memuncak hanya beberapa hari sebelum berangkat (median H-1 sampai H-5, tujuh musim berturut-turut).

In [5]:
# metrik resmi temuan, seed terkunci
from numpy.random import default_rng

rng = default_rng(2026)

def ci_median(x, n_boot=10000):
    x = np.asarray(x, float)
    sm = np.array([np.median(rng.choice(x, len(x))) for _ in range(n_boot)])
    return float(np.median(x)), float(np.percentile(sm, 2.5)), float(np.percentile(sm, 97.5))

metrik = {"aksi": {}, "niat": {}}

h_pergi = int(harian.loc[:0].idxmax()); h_balik = int(harian.loc[1:].idxmax())
metrik["aksi"]["puncak_pergi"] = {"h": h_pergi, "total": int(harian[h_pergi])}
metrik["aksi"]["puncak_balik"] = {"h": h_balik, "total": int(harian[h_balik])}
metrik["aksi"]["tersepi_jendela"] = int(harian.min())
metrik["aksi"]["rasio_puncak_vs_tersepi"] = float(harian.max() / harian.min())
metrik["aksi"]["jendela_pergi"] = int(harian.loc[-10:-1].sum())
metrik["aksi"]["jendela_balik"] = int(harian.loc[1:].sum())
metrik["aksi"]["cekungan_tahunbaru_min_H5H7"] = int(harian.loc[5:7].min())
per_moda = {}
for moda, d in k.groupby("moda"):
    i = d["penumpang"].idxmax()
    per_moda[moda] = {"total": int(d["penumpang"].sum()),
                      "h_puncak": int(d.loc[i, "h"]), "maks": int(d.loc[i, "penumpang"])}
metrik["aksi"]["per_moda"] = per_moda

for term in FRASA:
    df3 = LIPAT[term].loc[-55:12].rolling(3, min_periods=2, center=True).mean()
    up, lag = [], []
    for m in MUSIM:
        w = df3[m].loc[-7:-1].dropna()
        up.append(float(w.max()))
        lag.append(float(w.idxmax()))
    mu, lo, hi = ci_median(up)
    mm, lo2, hi2 = ci_median(lag)
    metrik["niat"][term] = {"median_lipat": mu, "ci": [lo, hi],
                            "median_rel_puncak": mm, "ci_lag": [lo2, hi2],
                            "lipat_per_musim": [round(u, 2) for u in up],
                            "lag_per_musim": [int(x) for x in lag]}

print(json.dumps(metrik, indent=2))
Path("data/metrik-006.json").write_text(json.dumps(metrik, indent=2))

{
  "aksi": {
    "puncak_pergi": {
      "h": -3,
      "total": 1011710
    },
    "puncak_balik": {
      "h": 4,
      "total": 1014842
    },
    "tersepi_jendela": 622820,
    "rasio_puncak_vs_tersepi": 1.629430654121576,
    "jendela_pergi": 8493265,
    "jendela_balik": 8910569,
    "cekungan_tahunbaru_min_H5H7": 754461,
    "per_moda": {
      "Angkutan Jalan": {
        "total": 3934367,
        "h_puncak": 4,
        "maks": 230527
      },
      "Angkutan Laut": {
        "total": 1750511,
        "h_puncak": -3,
        "maks": 102286
      },
      "Angkutan Perkeretaapian": {
        "total": 4357747,
        "h_puncak": 4,
        "maks": 245982
      },
      "Angkutan SDP": {
        "total": 2917681,
        "h_puncak": -3,
        "maks": 178599
      },
      "Angkutan Udara": {
        "total": 5359093,
        "h_puncak": -3,
        "maks": 301488
      }
    }
  },
  "niat": {
    "tiket kereta api": {
      "median_lipat": 1.837677944167129,
      "ci": [
    

2368

## Batas & cara reproduksi

Lima hal perlu dibilang. Pertama, data perjalanan nyata hanya tersedia untuk satu event Nataru (2024/25) di katalog terbuka; bentuk dua puncak dibaca dari 21 hari itu, bukan rerata banyak tahun. Kedua, angka Posko adalah penumpang-hari lintas wilayah terpantau; katalognya menyatakan data di luar H-7 s.d. H+10 belum direkonsiliasi, dan angka rekap yang tercetak di berita memakai penyaringan tersendiri (misalnya 17,18 juta angkutan umum) sehingga berbeda sedikit dari jumlah tabel ini. Ketiga, indeks Google Trends relatif per musim dan bukan jumlah orang; perbandingan antar musim hanya dibaca pada bentuknya. Keempat, mencari tiket bukan berarti pergi, jadi dua sumber tidak dibaur jadi satu angka.
Kelima, jendela Posko mencakup akhir pekan dan hari libur; denyut hariannya
bukan pola hari kerja murni.

In [6]:
def kartu_teks():
    gaya.simpan(gaya.kartu("linkedin", "EDISI 006 · WAKTU · JANUARI 2027",
        "Puncak perjalanan Nataru:\nberangkat atau balik?",
        [(gaya.INK2, "Tiap tahun beritanya:\npuncak arus mudik hari ini.\nSaya buka data Posko Nataru\nKemenhub dan tujuh musim\npencarian tiket."),
         (gaya.AKSEN, "(jawabannya di dalam)")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-01-pertanyaan", "linkedin", penuh=True)

    gaya.simpan(gaya.kartu("linkedin", "EDISI 006 · WAKTU",
        "Datanya dari mana?",
        [(gaya.INK, "Posko Nataru 2024/25 Kemenhub:\n5 moda, 21 hari (H-10 s.d. H+10),\n18,3 juta penumpang-hari."),
         (gaya.INK2, "Niat dibaca dari Google Trends:\ntiket kereta api, pesawat, bus,\n7 musim; tiap musim dinyatakan\nsebagai lipatan dasar sendiri.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-02-data", "linkedin", penuh=True)

    gaya.simpan(gaya.kartu("linkedin", "EDISI 006 · WAKTU",
        "Temuan dan batasnya",
        [(gaya.AKSEN, "Bukan satu lonjakan: dua puncak,\npergi di H-3 dan balik di H+4,\nmasing-masing 1,01 juta/hari."),
         (gaya.INK2, "Batas: perjalanan nyata baru satu\nNataru yang terbuka; dan mencari\ntiket bukan berarti pergi, jadi\ndua sumber tidak dibaur satu angka.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-05-batas", "linkedin", penuh=True)


kartu_teks()